## Test AMP implementation

In [1]:
from WERM.config import unif_prior_to_η_ϕ
from WERM.config import η_to_ψ
from WERM.signal_priors import GaussianSignal
from WERM.model import *
import WERM
import numpy as np
import jax.numpy as jnp # Not used in any randomness, all randomness is in numpy
import jax
import numpy.random as nprandom
import matplotlib.pyplot as plt
import scipy
from functools import partial
from tqdm import tqdm
tqdm = partial(tqdm, position=0, leave=True)

p = 500
L = 2 # num signals. L-1 changepoints

δ_list = [2.0, 2.5, 3.0, 3.5, 4.0]

# chgpt_locations = np.array([0.3, 0.7])
chgpt_locations = np.array([0.4])

Σ = np.zeros((p, p))
for i in range(p):
    for j in range(p):
        Σ[i, j] = 0.5 ** (np.abs(i - j))
L_chol = scipy.linalg.cholesky(Σ, lower = True)

def ℓ(Θ, y):
    return ζ(Θ) - y * Θ

for i in tqdm(range(len(δ_list))):
    δ = δ_list[i]
    print("δ = ", δ)

    n = int(δ * p)
    true_chgpt_locations = lambda n: (n*chgpt_locations).astype(int)
    C_true = η_to_ψ(true_chgpt_locations(n), n)

    # --- Preliminaries ---
    # Changepoint locations
    Δ = lambda n: int(n/100)
    Lmin = 2
    Lmax = 2
    # p_l = 1/L * np.ones(L) # puts equal weight on all number of signals, set None otherwise. Best for estimating size. 
    p_l = None
    num_valid_configs, η_arr, p_η_arr, ϕ = unif_prior_to_η_ϕ(Lmin, Lmax, Δ(n), n, p_l = p_l) # Lxn matrix
    assert η_arr.shape == (num_valid_configs, Lmax-1) and \
        len(p_η_arr) == num_valid_configs
    assert ϕ.shape == (Lmax, n)

    ℓ_list = np.array([(lambda θ, y: ϕ[0, i] * ℓ(θ, y)) for i in range(n)])

    # Gaussian Signal Prior
    κ = 7.0
    B̃_cov = κ**2 * δ * np.eye(L) # This technically keeps the signal power fixed, this seems important for estimating size. 
    signal_prior = GaussianSignal(B̃_cov)

    B̃ = signal_prior.sample(p)

    # Generate the observation vector Y
    X = np.random.normal(0, np.sqrt(1/n), (n, p)) @ L_chol.T

    # scaler = StandardScaler().fit(X)
    # X = scaler.transform(X)
    Θ = X @ B̃
    y = LR(Θ, C_true)

    β̂, θ̂ = WERM.amp(X, y, n, p, ℓ_list, max_iter = 10)

  0%|          | 0/5 [00:00<?, ?it/s]

δ =  2.0


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

θ.shape:  (1000,)
y.shape:  (1000, 1)


TypeError: 'BatchTracer' object is not callable

<function __main__.<listcomp>.<lambda>(θ, y)>